In [1]:
import numpy as np
import pandas as pd
import pyhocon
from sklearn.cluster import AgglomerativeClustering
from torch.utils import data
import torch
from tqdm import tqdm

from dataset import CrossEncoderDatasetFull,CrossEncoderMixedDatasetFull
from cluster_documents import return_mentions_with_predicted_topics
from models import FullCrossEncoder

from collections import Counter



/home/sago/miniconda3/envs/pgkpr/lib/python3.8/site-packages/transformers/generation_utils.py:24: FutureWarning: Importing `GenerationMixin` from `src/transformers/generation_utils.py` is deprecated and will be removed in Transformers v5. Import as `from transformers import GenerationMixin` instead.
  warnings.warn(
Xformers is not installed correctly. If you want to use memorry_efficient_attention to accelerate training use the following command to install Xformers
pip install xformers.


In [2]:
#Load config files
config = pyhocon.ConfigFactory.parse_file('configs/config_pairwise.json')
# Load the test data

datasets_config = pyhocon.ConfigFactory.parse_file('configs/config_datasets.json')

datasets_config = pyhocon.ConfigFactory.parse_file('configs/config_datasets.json')
common_data_folder = datasets_config.get_string("common_data_folder")
for key in datasets_config:
    if isinstance(datasets_config[key], dict) and "data_folder" in datasets_config[key]:
        datasets_config[key]["data_folder"] = datasets_config[key]["data_folder"].replace("${common_data_folder}", common_data_folder)

config_experiments = pyhocon.ConfigFactory.parse_file('configs/config_experiments.json')

In [3]:
#Load model
device_ids = config.gpu_num
device = torch.device("cuda:{}".format(device_ids[0]))
cross_encoder_single = FullCrossEncoder(config).to(device)

# cross_encoder_single.model = RobertaModel.from_pretrained(os.path.join(config.model_path, 'bert')).to(device) # I have to choose the right model path
# cross_encoder_single.linear.load_state_dict(torch.load(os.path.join(config.model_path, 'linear')))
cross_encoder = torch.nn.DataParallel(cross_encoder_single, device_ids=device_ids)
cross_encoder.eval()

loading file vocab.json from cache at /home/sago/.cache/huggingface/hub/models--roberta-large/snapshots/716877d372b884cad6d419d828bac6c85b3b18d9/vocab.json
loading file merges.txt from cache at /home/sago/.cache/huggingface/hub/models--roberta-large/snapshots/716877d372b884cad6d419d828bac6c85b3b18d9/merges.txt
loading file added_tokens.json from cache at None
loading file special_tokens_map.json from cache at None
loading file tokenizer_config.json from cache at None
loading configuration file config.json from cache at /home/sago/.cache/huggingface/hub/models--roberta-large/snapshots/716877d372b884cad6d419d828bac6c85b3b18d9/config.json
Model config RobertaConfig {
  "_name_or_path": "roberta-large",
  "architectures": [
    "RobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 

DataParallel(
  (module): FullCrossEncoder(
    (model): RobertaModel(
      (embeddings): RobertaEmbeddings(
        (word_embeddings): Embedding(50267, 1024)
        (position_embeddings): Embedding(514, 1024, padding_idx=1)
        (token_type_embeddings): Embedding(1, 1024)
        (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): RobertaEncoder(
        (layer): ModuleList(
          (0-23): 24 x RobertaLayer(
            (attention): RobertaAttention(
              (self): RobertaSelfAttention(
                (query): Linear(in_features=1024, out_features=1024, bias=True)
                (key): Linear(in_features=1024, out_features=1024, bias=True)
                (value): Linear(in_features=1024, out_features=1024, bias=True)
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (output): RobertaSelfOutput(
                (dense): Linear(in_features=1024

In [4]:
# init train and dev set
if "Experiment" in config.experiment_id:
    test = CrossEncoderMixedDatasetFull(config,datasets_config,config_experiments[config.experiment_id]['test'])
    
else:
    test = CrossEncoderDatasetFull(config, 'test')

    predict_topics = True #TODO Pass by config parameter
    if predict_topics:
        mentions_w_predicted_topics = return_mentions_with_predicted_topics(test)
        test.assign_mentions_with_new_topics(mentions_w_predicted_topics)

    test.create_mention_pairs_by_topics(use_predicted=True)
    test.prepare_pair_of_mentions()


test_loader = data.DataLoader(test, batch_size=config.batch_size, shuffle=False)

NUM_TOPICS 5 NUM_MENTIONS 2622


/home/sago/miniconda3/envs/pgkpr/lib/python3.8/site-packages/sklearn/cluster/_kmeans.py:1366: FutureWarning: algorithm='auto' is deprecated, it will be removed in 1.3. Using 'lloyd' instead.
  warnings.warn(
/home/sago/Documents/GitHub/CDCR_lexical_diversity/MTL/cross_encoder_adapted/dataset.py:138: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(labels,dtype=torch.float)


In [6]:
test.mentions[0].keys()

dict_keys(['coref_chain', 'mention_ner', 'mention_head_pos', 'mention_head_lemma', 'mention_head', 'mention_head_id', 'doc_id', 'doc', 'is_continuous', 'is_singleton', 'mention_id', 'mention_type', 'mention_full_type', 'score', 'sent_id', 'mention_context', 'tokens_number_context', 'tokens_number', 'tokens_str', 'tokens_text', 'topic_id', 'topic', 'subtopic_id', 'subtopic', 'coref_type', 'description', 'conll_doc_key', 'predicted_topic'])

In [16]:
test.mentions[1]['tokens_number']

[7, 8, 9]

In [17]:
test.mentions[1]['tokens_str']

'in western Canada'

In [19]:
test.mentions[1]['tokens_number_context']

[15, 16, 17]

In [20]:
test.mentions[1]['mention_context']

['2',
 'leaders',
 'of',
 'polygamist',
 'group',
 'arrested',
 'in',
 'Canada',
 'Two',
 'top',
 'leaders',
 'of',
 'a',
 'polygamous',
 'community',
 'in',
 'western',
 'Canada',
 'have',
 'been',
 'arrested',
 'and',
 'charged',
 'with',
 'practicing',
 'polygamy',
 ',',
 'British',
 'Columbia',
 "'s",
 'attorney',
 'general',
 'said',
 'Wednesday',
 'Attorney',
 'General',
 'Wally',
 'Oppal',
 'said',
 'Winston',
 'Blackmore',
 'is',
 'charged',
 'with',
 'marrying',
 '20',
 'women',
 ',',
 'while',
 'James',
 'Oler',
 'is',
 'accused',
 'of',
 'marrying',
 'two',
 'women',
 '.',
 '``',
 'This',
 'has',
 'been',
 'a',
 'very',
 'complex',
 'issue',
 ',',
 "''",
 'Oppal',
 'said',
 '.',
 '``',
 'It',
 "'s",
 'been',
 'with',
 'us',
 'for',
 'well',
 'over',
 '20',
 'years',
 '.',
 "''",
 'Blackmore',
 ',',
 'long',
 'known',
 'as',
 '``',
 'the',
 'Bishop',
 'of',
 'Bountiful',
 ',',
 "''",
 'runs',
 'an',
 'independent',
 'sect',
 'of',
 'about',
 '400',
 'members',
 'in',
 'the',


In [6]:
mentions_dictionary = {d["mention_id"]: d for d in test.mentions}
df_score = pd.DataFrame({"mention1":[id.split("[SEP]")[1] for id in test.instance_ids],
              "mention2":[id.split("[SEP]")[2] for id in test.instance_ids],
              "similarity_score":torch.rand(len(test.instance_ids))})
df_score["predicted_topic"] = df_score["mention1"].map(mentions_dictionary).apply(lambda x: x.get("predicted_topic") if x else None)

In [7]:
df_read = pd.read_csv("predicted_clusters.csv")

In [8]:
# Create distance matrix
def create_distance_matrix(df):

    mention_to_index = {mention: index for index, mention in enumerate(pd.concat([df['mention1'], df['mention2']]).unique())}
    index_to_mention = {index: mention for index, mention in enumerate(pd.concat([df['mention1'], df['mention2']]).unique())}
    # Create an empty distance matrix
    num_mentions = len(mention_to_index)
    distance_matrix = np.ones((num_mentions, num_mentions))


    # Convert mentions to indices
    indices1 = df['mention1'].map(mention_to_index)
    indices2 = df['mention2'].map(mention_to_index)

    # Calculate the distances using vectorized operations
    distance_matrix[indices1, indices2] = 1 - df['similarity_score']
    distance_matrix[indices2, indices1] = 1 - df['similarity_score']

    return distance_matrix, index_to_mention

In [21]:
dict_distance_matrices_topic = {}
dict_index_to_mention_topic = {}
for topic in df_read.predicted_topic1.unique():
    df_temp = df_read[df_read["predicted_topic1"]== topic]
    dict_distance_matrices_topic[topic],dict_index_to_mention_topic[topic]  =create_distance_matrix(df_temp)

In [11]:
def perform_variable_threshold_agglomerative(distance_matrix, thresholds,linkage):
    """
    Perform agglomerative clustering with varying distance thresholds.
    
    Parameters:
        distance_matrix (numpy.ndarray): The distance matrix between data points.
        thresholds (list): List of distance thresholds for clustering.
        
    Returns:
        clusters (list): List of lists containing cluster labels for each threshold.
    """
    clustering = [
        AgglomerativeClustering(n_clusters=None, affinity='precomputed', linkage=linkage,
                                distance_threshold=threshold)
        for threshold in thresholds
    ]
    
    clusters = [[] for _ in thresholds]
    max_ids = [0] * len(thresholds)

    for i, agglomerative in enumerate(clustering):
        predicted = agglomerative.fit(distance_matrix)
        predicted_clusters = predicted.labels_ + max_ids[i]
        max_ids[i] = max(predicted_clusters) + 1
        clusters[i].extend(predicted_clusters)
    
    return clusters




In [42]:
# Creating clusters
dict_clusters_average_linkage = {}
dict_clusters_complete_linkage = {}

thresholds = [0.3, 0.5, 0.55, 0.6, 0.65, 0.7]
for topic, distance_mat in dict_distance_matrices_topic.items():
    dict_clusters_average_linkage[topic] = perform_variable_threshold_agglomerative(distance_mat, thresholds,linkage="average")
    dict_clusters_complete_linkage[topic] = perform_variable_threshold_agglomerative(distance_mat, thresholds,linkage="complete")



/home/sago/miniconda3/envs/pgkpr/lib/python3.8/site-packages/sklearn/cluster/_agglomerative.py:983: FutureWarning: Attribute `affinity` was deprecated in version 1.2 and will be removed in 1.4. Use `metric` instead
  warnings.warn(
/home/sago/miniconda3/envs/pgkpr/lib/python3.8/site-packages/sklearn/cluster/_agglomerative.py:983: FutureWarning: Attribute `affinity` was deprecated in version 1.2 and will be removed in 1.4. Use `metric` instead
  warnings.warn(
/home/sago/miniconda3/envs/pgkpr/lib/python3.8/site-packages/sklearn/cluster/_agglomerative.py:983: FutureWarning: Attribute `affinity` was deprecated in version 1.2 and will be removed in 1.4. Use `metric` instead
  warnings.warn(
/home/sago/miniconda3/envs/pgkpr/lib/python3.8/site-packages/sklearn/cluster/_agglomerative.py:983: FutureWarning: Attribute `affinity` was deprecated in version 1.2 and will be removed in 1.4. Use `metric` instead
  warnings.warn(
/home/sago/miniconda3/envs/pgkpr/lib/python3.8/site-packages/sklearn/clu

/home/sago/miniconda3/envs/pgkpr/lib/python3.8/site-packages/sklearn/cluster/_agglomerative.py:983: FutureWarning: Attribute `affinity` was deprecated in version 1.2 and will be removed in 1.4. Use `metric` instead
  warnings.warn(
/home/sago/miniconda3/envs/pgkpr/lib/python3.8/site-packages/sklearn/cluster/_agglomerative.py:983: FutureWarning: Attribute `affinity` was deprecated in version 1.2 and will be removed in 1.4. Use `metric` instead
  warnings.warn(
/home/sago/miniconda3/envs/pgkpr/lib/python3.8/site-packages/sklearn/cluster/_agglomerative.py:983: FutureWarning: Attribute `affinity` was deprecated in version 1.2 and will be removed in 1.4. Use `metric` instead
  warnings.warn(
/home/sago/miniconda3/envs/pgkpr/lib/python3.8/site-packages/sklearn/cluster/_agglomerative.py:983: FutureWarning: Attribute `affinity` was deprecated in version 1.2 and will be removed in 1.4. Use `metric` instead
  warnings.warn(
/home/sago/miniconda3/envs/pgkpr/lib/python3.8/site-packages/sklearn/clu

In [43]:
#Metrics

def muc(clusters, mention_to_gold):
    tp, p = 0, 0
    for c in clusters:
        p += len(c) - 1
        tp += len(c)
        linked = set()
        for m in c:
            if m in mention_to_gold:
                linked.add(mention_to_gold[m])
            else:
                tp -= 1
        tp -= len(linked)
    return tp, p

def b_cubed(clusters, mention_to_gold):
    num, dem = 0, 0

    for c in clusters:
        gold_counts = Counter()
        correct = 0
        for m in c:
            if m in mention_to_gold:
                gold_counts[mention_to_gold[m]] += 1
        for c2 in gold_counts:
            correct += gold_counts[c2] * gold_counts[c2]

        num += correct / float(len(c))
        dem += len(c)

    return num, dem

def lea(input_clusters, output_clusters, mention_to_gold):
    num, dem = 0, 0

    for c in input_clusters:
        if len(c) == 1:
            all_links = 1
            if c[0] in mention_to_gold and len(output_clusters[mention_to_gold[c[0]]]) == 1:
                common_links = 1
            else:
                common_links = 0
        else:
            common_links = 0
            all_links = len(c) * (len(c) - 1) / 2.0
            for i, m in enumerate(c):
                if m in mention_to_gold:
                    for m2 in c[i + 1:]:
                        if m2 in mention_to_gold and mention_to_gold[m] == mention_to_gold[m2]:
                            common_links += 1

        num += len(c) * common_links / float(all_links)
        dem += len(c)

    return num, dem

In [44]:

def calculate_and_print_indicators(mentions_dictionary, index_to_mention, clusters):
    indices = []
    coref_chains = []
    predicted_clusters = []
    for i, cluster in enumerate(clusters[0]):
        indices.append(i)
        coref_chains.append(mentions_dictionary[index_to_mention[i]]["coref_chain"])
        predicted_clusters.append(cluster)

    df_predicted_coref = pd.DataFrame({"index": indices, "coref_chain": coref_chains, "predicted_cluster": predicted_clusters})

    coref_to_true = df_predicted_coref.groupby('coref_chain')['predicted_cluster'].first().to_dict()
    coref_to_predicted = df_predicted_coref.groupby('coref_chain')['predicted_cluster'].apply(list).to_dict()

    muc_tp, muc_p = muc(list(coref_to_predicted.values()), coref_to_true)
    muc_recall = muc_tp / muc_p if muc_p > 0 else 0
    print(f"MUC Recall: {muc_recall:.4f}")

    b_cubed_num, b_cubed_den = b_cubed(list(coref_to_predicted.values()), coref_to_true)
    b_cubed_precision = b_cubed_num / b_cubed_den if b_cubed_den > 0 else 0
    print(f"B-Cubed Precision: {b_cubed_precision:.4f}")

    lea_num, lea_den = lea(list(coref_to_predicted.values()), list(coref_to_predicted.values()), coref_to_true)
    lea_precision = lea_num / lea_den if lea_den > 0 else 0
    print(f"LEA Precision: {lea_precision:.4f}")



In [45]:
dict_clusters_average_linkage
dict_clusters_complete_linkage


{1: [[20,
   120,
   140,
   85,
   15,
   60,
   45,
   114,
   33,
   93,
   38,
   114,
   81,
   168,
   26,
   156,
   172,
   148,
   88,
   90,
   15,
   48,
   74,
   158,
   87,
   84,
   94,
   50,
   35,
   68,
   177,
   169,
   167,
   34,
   147,
   28,
   59,
   52,
   5,
   97,
   133,
   130,
   28,
   51,
   69,
   146,
   79,
   64,
   140,
   94,
   73,
   0,
   57,
   20,
   117,
   76,
   75,
   34,
   31,
   47,
   171,
   71,
   110,
   43,
   17,
   115,
   100,
   147,
   72,
   27,
   2,
   175,
   8,
   46,
   85,
   116,
   36,
   72,
   176,
   77,
   105,
   89,
   56,
   57,
   43,
   116,
   53,
   129,
   67,
   84,
   83,
   40,
   105,
   126,
   30,
   30,
   42,
   5,
   127,
   41,
   6,
   96,
   174,
   58,
   14,
   18,
   27,
   117,
   24,
   148,
   22,
   54,
   84,
   15,
   16,
   48,
   134,
   39,
   44,
   67,
   60,
   160,
   34,
   15,
   96,
   89,
   104,
   94,
   45,
   99,
   26,
   21,
   91,
   68,
   60,
   123,
   13,
   12

In [46]:
dict_clusters_average_linkage.keys()

dict_keys([1, 2, 0, 3, 4])

In [47]:
dict_index_to_mention_topic

{1: {0: '36_1ecb_LOC17642835073942721_0_6_MJgD',
  1: '36_1ecb_LOC17642835073942721_1_7_CchK',
  2: '36_2ecb_LOC17642835073942721_0_3_MDAh',
  3: '36_8ecb_LOC17642835073942721_1_12_9ymg',
  4: '36_8ecb_LOC17642835073942721_0_3_57id',
  5: '36_9ecb_LOC17642835073942721_0_4_RaCR',
  6: '36_1ecb_HUM18520763746825423_1_6_WbKP',
  7: '36_1ecb_HUM18520763746825423_0_3_J9om',
  8: '36_2ecb_HUM18520763746825423_1_4_i5yY',
  9: '36_3ecb_HUM18520763746825423_1_7_26xC',
  10: '36_7ecb_HUM18520763746825423_1_13_fta9',
  11: '36_7ecb_HUM18520763746825423_0_6_gMJX',
  12: '36_8ecb_HUM18520763746825423_1_4_GFmR',
  13: '36_1ecb_TIM17720510666890266_1_25_MZAJ',
  14: '36_3ecb_TIM17720510666890266_2_14_Nw58',
  15: '36_5ecb_TIM17720510666890266_1_27_657k',
  16: '36_6ecb_TIM17720510666890266_1_9_dweX',
  17: '36_7ecb_TIM17720510666890266_2_8_URXL',
  18: '36_1ecb_HUM17720489125766418_1_22_homB',
  19: '36_3ecb_HUM17720489125766418_2_3_TEdR',
  20: '36_5ecb_HUM17720489125766418_1_20_X6Zu',
  21: '36_6ec

In [48]:
# Example usage
# Assuming mentions_dictionary, index_to_mention, and clusters are defined
for topic, index in dict_index_to_mention_topic.items():
    calculate_and_print_indicators(mentions_dictionary, index, dict_clusters_average_linkage[topic])
    calculate_and_print_indicators(mentions_dictionary, index, dict_clusters_complete_linkage[topic])

MUC Recall: 0.0000
B-Cubed Precision: 0.0000
LEA Precision: 0.0000
MUC Recall: 0.0000
B-Cubed Precision: 0.0000
LEA Precision: 0.0000
MUC Recall: 0.0000
B-Cubed Precision: 0.0000
LEA Precision: 0.0000
MUC Recall: 0.0000
B-Cubed Precision: 0.0000
LEA Precision: 0.0000
MUC Recall: 0.0000
B-Cubed Precision: 0.0000
LEA Precision: 0.0000
MUC Recall: 0.0000
B-Cubed Precision: 0.0000
LEA Precision: 0.0000
MUC Recall: 0.0000
B-Cubed Precision: 0.0000
LEA Precision: 0.0000
MUC Recall: 0.0000
B-Cubed Precision: 0.0000
LEA Precision: 0.0000
MUC Recall: 0.0000
B-Cubed Precision: 0.0000
LEA Precision: 0.0000
MUC Recall: 0.0000
B-Cubed Precision: 0.0000
LEA Precision: 0.0000
